# ⚕️ Dual Diagnosis RAG — اجرای یک‌کلیکی در Google Colab

این نوت‌بوک:
1. کد را از GitHub می‌گیرد؛ 2. مقالات اعتبارسنجی‌شده را از Hugging Face دریافت می‌کند؛ 3. PDF/کتاب مجاز شما را می‌پذیرد؛ 4. ایندکس RAG را می‌سازد؛ 5. پرسش آزمایشی را اجرا می‌کند؛ 6. در صورت تنظیم کلید، نتیجه را در W&B ثبت می‌کند.

> **ایمنی:** ابزار آموزشی/کمک‌تصمیم است، جایگزین پزشک نیست. فقط فایل‌هایی را وارد کنید که اجازه استفاده از آن‌ها را دارید. اطلاعات هویتی بیمار را آپلود نکنید.


## 1) بررسی محیط و دریافت پروژه

In [ ]:
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB)
!rm -rf /content/dual-diagnosis-rag
!git clone -q https://github.com/SBZ-EDU/dual-diagnosis-rag.git /content/dual-diagnosis-rag
%cd /content/dual-diagnosis-rag
print('Repository ready')


## 2) نصب وابستگی‌های سبک RAG

In [ ]:
!pip -q install "sentence-transformers>=3,<4" "transformers>=4.44,<5" "huggingface_hub>=0.27,<2" "pypdf>=5,<6" "wandb>=0.18,<1" "peft>=0.12,<1" "accelerate>=0.30" gradio
print('Dependencies installed')


## 3) دریافت دیتاست مقالات از Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, json
repo='sosa123454321/dual-diagnosis-dataset'
files={
 'articles/validated_articles.jsonl':'data/articles/validated_articles.jsonl',
 'articles/international_100.jsonl':'data/articles/international_100.jsonl',
 'articles/iran_affiliated_100.jsonl':'data/articles/iran_affiliated_100.jsonl',
 'guidelines/addiction_guidelines.jsonl':'data/guidelines/addiction_guidelines.jsonl',
}
for remote,local in files.items():
    cached=hf_hub_download(repo_id=repo,repo_type='dataset',filename=remote)
    dst=Path(local); dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy(cached,dst)
    print('✓',remote,'->',local)
def count_jsonl(path): return sum(1 for x in open(path,encoding='utf-8') if x.strip())
print({'validated':count_jsonl('data/articles/validated_articles.jsonl'),
       'international':count_jsonl('data/articles/international_100.jsonl'),
       'iran_affiliated':count_jsonl('data/articles/iran_affiliated_100.jsonl'),
       'guidelines':count_jsonl('data/guidelines/addiction_guidelines.jsonl')})


## 4) دانلود خودکار PDFهای Open Access و استخراج متن
این بخش فقط URLهایی را دریافت می‌کند که در Manifest به‌عنوان Open Access ثبت شده‌اند؛ Paywall دور زده نمی‌شود. برای جلوگیری از پرشدن فضای Colab، سقف تعداد و حجم قابل تنظیم است.

In [ ]:
import requests, hashlib, time
from pathlib import Path
manifest_cache=hf_hub_download(repo_id=repo,repo_type='dataset',filename='articles/open_access_pdf_manifest.jsonl')
manifest=[json.loads(x) for x in open(manifest_cache,encoding='utf-8') if x.strip() and json.loads(x).get('pdf_url')]
MAX_FILES=50          # برای همه لینک‌ها تا 128 افزایش دهید
MAX_TOTAL_MB=200      # سقف دانلود
PDF_DIR=Path('/content/oa_pdfs'); PDF_DIR.mkdir(exist_ok=True)
TEXT_DIR=Path('data/articles/oa_fulltext'); TEXT_DIR.mkdir(parents=True,exist_ok=True)
selected=[]; planned=0
for x in manifest:
    size=x.get('content_length_bytes') or 0
    if len(selected)>=MAX_FILES: break
    if size and planned+size>MAX_TOTAL_MB*1024*1024: continue
    selected.append(x); planned+=size
print('Manifest PDFs:',len(manifest),'Selected:',len(selected),'Known planned MB:',round(planned/1024/1024,2))


In [ ]:
from pypdf import PdfReader
report=[]; downloaded=0
for i,x in enumerate(selected,1):
    url=x['pdf_url']; key=(x.get('doi') or x.get('openalex_id') or str(i)).rsplit('/',1)[-1]
    safe=hashlib.sha256(key.encode()).hexdigest()[:16]
    pdf=PDF_DIR/f'{safe}.pdf'; txt=TEXT_DIR/f'{safe}.md'
    try:
        with requests.get(url,stream=True,timeout=45,headers={'User-Agent':'Mozilla/5.0 OpenAccessResearchBot/1.0'}) as r:
            r.raise_for_status(); ctype=r.headers.get('content-type','')
            if 'pdf' not in ctype.lower() and not url.lower().endswith('.pdf'): raise ValueError('response is not PDF')
            with open(pdf,'wb') as f:
                for chunk in r.iter_content(1024*1024):
                    if chunk:
                        if downloaded+len(chunk)>MAX_TOTAL_MB*1024*1024: raise RuntimeError('download cap reached')
                        f.write(chunk); downloaded+=len(chunk)
        reader=PdfReader(str(pdf)); text='\n'.join((page.extract_text() or '') for page in reader.pages)
        txt.write_text(f"# {x.get('title','Open Access Paper')}\n\nSource: {url}\nDOI: {x.get('doi')}\n\n{text}",encoding='utf-8')
        report.append({'title':x.get('title'),'url':url,'pdf':str(pdf),'text':str(txt),'bytes':pdf.stat().st_size,'pages':len(reader.pages),'text_chars':len(text),'status':'ok'})
        print(f"[{i}/{len(selected)}] ✓",x.get('title','')[:65],len(reader.pages),'pages')
    except Exception as e:
        if pdf.exists() and not txt.exists(): pdf.unlink()
        report.append({'title':x.get('title'),'url':url,'status':'failed','error':str(e)[:300]})
        print(f"[{i}/{len(selected)}] ✗",str(e)[:100])
Path('/content/pdf_download_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
ok=[r for r in report if r['status']=='ok']
print({'successful':len(ok),'failed':len(report)-len(ok),'downloaded_mb':round(downloaded/1024/1024,2),'extracted_characters':sum(r.get('text_chars',0) for r in ok)})


## 5) اختیاری: آپلود PDF مقاله/کتاب مجاز
PDFها به متن محلی تبدیل می‌شوند و فایل اصلی به Hugging Face یا W&B ارسال نمی‌شود مگر خودتان صریحاً این کار را انجام دهید.

In [ ]:
from pathlib import Path
if IN_COLAB:
    from google.colab import files
    uploaded=files.upload()  # اگر فایل ندارید این سلول را اجرا نکنید
    from pypdf import PdfReader
    out=Path('data/articles')
    for name,data in uploaded.items():
        if not name.lower().endswith('.pdf'): continue
        Path(name).write_bytes(data)
        reader=PdfReader(name)
        text='\n'.join((p.extract_text() or '') for p in reader.pages)
        safe=Path(name).stem.replace(' ','_')+'.md'
        (out/safe).write_text(f'# {Path(name).stem}\n\n{text}',encoding='utf-8')
        print(name,'->',out/safe,len(text),'characters')
else: print('Upload is available in Colab.')


## 6) ساخت ایندکس برداری RAG

In [ ]:
import os
os.environ['EMBED_MODEL']='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
!python -m scripts.build_index
from rag import store
chunks,vectors=store.load()
print('Chunks:',len(chunks),'Vector shape:',vectors.shape)


## 7) تست بازیابی مقاله

In [ ]:
from rag import retriever
question='در روان‌پریشی همراه مصرف مواد، در صورت لغزش چه ارزیابی‌هایی لازم است؟'
hits=retriever.search(question,top_k=5)
for i,h in enumerate(hits,1):
    print(f"\n[{i}] score={h['score']:.3f} source={h['source']}\n{h['text'][:700]}")


## 8) پاسخ RAG
حالت پیش‌فرض استخراجی و رایگان است. برای مدل مولد، GPU را از Runtime → Change runtime type فعال کنید.

In [ ]:
os.environ['USE_GENERATOR']='0'  # برای Colab رایگان سریع و قابل اتکا
from rag import pipeline
result=pipeline.answer(question)
print(result['answer'])
print('\nSources:')
for s in result['sources']: print('-',s)


## 9) ارزیابی خودکار سلامت RAG
این تست بررسی می‌کند که ایندکس خالی نباشد، ابعاد بردار درست باشد، منابع مقاله/راهنما وارد شده باشند و برای چند پرسش نمونه نتیجه برگردد. این آزمون فنی است، نه اعتبارسنجی بالینی.

In [ ]:
from collections import Counter
assert len(chunks) >= 200, f'Expected >=200 chunks, got {len(chunks)}'
assert vectors.shape[0] == len(chunks), 'Chunk/vector count mismatch'
assert vectors.shape[1] > 100, 'Embedding dimension looks invalid'
types=Counter(c.get('type') for c in chunks)
assert types['article'] > 0, 'Articles missing from index'
assert types['guideline'] > 0, 'Guidelines missing from index'
tests=[
 'درمان اختلال مصرف مواد افیونی',
 'روان درمانی در تشخیص دوگانه',
 'کاهش آسیب و پیشگیری از اوردوز',
 'روان پریشی همراه مصرف کانابیس',
]
scores=[]
for query in tests:
    found=retriever.search(query,top_k=3)
    assert found, f'No retrieval result for: {query}'
    scores.append(float(found[0]['score']))
    print('✓',query,'top=',round(scores[-1],3),'source=',found[0]['source'])
print({'status':'PASS','chunks':len(chunks),'dimensions':vectors.shape[1],
       'types':dict(types),'mean_top_score':sum(scores)/len(scores)})


## 10) ثبت ارزیابی در W&B (اختیاری)
کلید را فقط در Colab Secrets با نام `WANDB_API_KEY` ذخیره کنید؛ آن را داخل نوت‌بوک ننویسید.

In [ ]:
import os
try:
    if IN_COLAB:
        from google.colab import userdata
        os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY') or ''
    if os.getenv('WANDB_API_KEY'):
        import wandb
        run=wandb.init(project='dual-diagnosis-rag',entity='elasa2next-sosa-',job_type='colab-rag-rebuild',config={'chunks':len(chunks),'embedding_model':os.environ['EMBED_MODEL'],'generator':False})
        wandb.log({'chunks_indexed':len(chunks),'top_retrieval_score':float(hits[0]['score']) if hits else 0,'sources_returned':len(result['sources'])})
        run.finish(); print(run.url)
    else: print('WANDB_API_KEY not set; logging skipped safely.')
except Exception as e: print('W&B skipped:',e)


## 11) Fine-tune اختیاری LoRA روی GPU
این مرحله وزن‌های پایه را تغییر نمی‌دهد؛ یک Adapter سبک می‌سازد. فقط داده‌های تأییدشده را برای آموزش استفاده کنید. مقاله خام بدون تبدیل به نمونه پرسش/پاسخ برای Fine-tune مناسب نیست.

In [ ]:
import torch
assert torch.cuda.is_available(), 'برای Fine-tune از Runtime → Change runtime type → T4 GPU استفاده کنید.'
# یک epoch برای تست؛ پس از ارزیابی می‌توانید افزایش دهید.
!USE_GENERATOR=0 python -m scripts.train --model Qwen/Qwen2.5-0.5B-Instruct --epochs 1 --batch 2 --out /content/model_lora


### انتشار Adapter در Hugging Face (اختیاری)
در Colab Secrets یک `HF_TOKEN` با دسترسی Write بسازید. خروجی Adapter مستقل از دیتابیس‌های RAG است.

In [ ]:
from huggingface_hub import HfApi
if IN_COLAB:
    from google.colab import userdata
    hf_token=userdata.get('HF_TOKEN')
else: hf_token=os.getenv('HF_TOKEN')
if hf_token:
    target='sosa123454321/dual-diagnosis-qwen-lora'
    api=HfApi(token=hf_token)
    api.create_repo(target,repo_type='model',exist_ok=True)
    api.upload_folder(repo_id=target,repo_type='model',folder_path='/content/model_lora',commit_message='Colab LoRA adapter')
    print('https://huggingface.co/'+target)
else: print('HF_TOKEN not set; adapter remains in /content/model_lora')


## 12) اجرای رابط Gradio در Colab (اختیاری)

In [ ]:
# اجرای این سلول یک لینک موقت Gradio می‌سازد.
# توجه: لینک با پایان نشست Colab خاموش می‌شود.
!USE_GENERATOR=0 GRADIO_SHARE=1 python app.py


## 13) ساخت و دانلود بسته خروجی
بسته شامل متن‌های استخراج‌شده، گزارش PDF و ایندکس برداری است. خود PDFها برای کاهش حجم ZIP نمی‌شوند؛ در `/content/oa_pdfs` باقی می‌مانند.

In [ ]:
import shutil, os
bundle='/content/dual_diagnosis_rag_output'
shutil.rmtree(bundle,ignore_errors=True); os.makedirs(bundle)
for src in ['index/chunks.json','index/vectors.npz','/content/pdf_download_report.json']:
    if os.path.exists(src): shutil.copy(src,bundle)
if TEXT_DIR.exists(): shutil.copytree(TEXT_DIR,Path(bundle)/'oa_fulltext',dirs_exist_ok=True)
archive=shutil.make_archive('/content/dual_diagnosis_rag_output','zip','/content',Path(bundle).name)
print(archive,round(os.path.getsize(archive)/1024/1024,2),'MB')
if IN_COLAB:
    from google.colab import files
    files.download(archive)


## خروجی‌ها
- ایندکس: `index/chunks.json` و `index/vectors.npz`
- مقالات: `data/articles/`
- داشبورد: https://wandb.ai/elasa2next-sosa-/dual-diagnosis-rag
- مخزن مدل: https://huggingface.co/sosa123454321/dual-diagnosis-rag
- دیتاست: https://huggingface.co/datasets/sosa123454321/dual-diagnosis-dataset
